In [1]:
# Hücre 1: İçe Aktarmalar ve Cihaz Seçimi
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.datasets import STL10
import torchvision.transforms as transforms
from torchvision.transforms.functional import rgb_to_grayscale
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# GPU varsa kullan, yoksa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [2]:
# Hücre 2: Veri Yükleme ve Ön İşleme
DATA_DIR = "./data"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Renkli dönüşüm
transform_color = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),      # [0,1], shape 3×128×128
])

# STL10 veri setlerini indir ve yükle
full_train = STL10(root=DATA_DIR, split='train', download=True, transform=transform_color)
test_set   = STL10(root=DATA_DIR, split='test',  download=True, transform=transform_color)

# Train/val ayrımı (%90/%10)
train_len = int(0.9 * len(full_train))
val_len   = len(full_train) - train_len
train_base, val_base = random_split(full_train, [train_len, val_len])

# Girdi–hedef çiftleri üreten sarıcı
class ColorizationDataset(Dataset):
    def __init__(self, base_ds):
        self.base = base_ds
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        color_img, _ = self.base[idx]                        # Tensor 3×128×128
        gray_img = rgb_to_grayscale(color_img, num_output_channels=1)
        return gray_img, color_img

train_ds = ColorizationDataset(train_base)
val_ds   = ColorizationDataset(val_base)
test_ds  = ColorizationDataset(test_set)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


100%|██████████| 2.64G/2.64G [05:46<00:00, 7.62MB/s]  


Train batches: 141, Val batches: 16, Test batches: 250


In [3]:
# Hücre 3: Model Tanımı
import torch.nn as nn
import torch.nn.functional as F

class ColorizationCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # --- Encoder ---
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 64,  kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 128×128×64
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 128×128×128
        self.pool1 = nn.MaxPool2d(2)                   # 64×64×128

        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 64×64×256
        self.pool2 = nn.MaxPool2d(2)                   # 32×32×256

        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 32×32×512
        self.conv5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 32×32×512

        # --- Decoder ---
        self.up1     = nn.Upsample(scale_factor=2, mode='nearest')  # →64×64×512
        self.dec1   = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 64×64×256
        self.dec2   = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 64×64×256

        self.up2     = nn.Upsample(scale_factor=2, mode='nearest')  # →128×128×256
        self.dec3   = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 128×128×128
        self.dec4   = nn.Sequential(
            nn.Conv2d(128,  64, kernel_size=3, padding=1), nn.ReLU(inplace=True)
        )  # 128×128×64

        # --- Output layer ---
        self.out_conv = nn.Conv2d(64, 3, kernel_size=3, padding=1)  # 128×128×3

    def forward(self, x):
        # Encoder
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.pool1(x)

        x = self.conv3(x)
        x = self.pool2(x)

        x = self.conv4(x)
        x = self.conv5(x)

        # Decoder
        x = self.up1(x)
        x = self.dec1(x)
        x = self.dec2(x)

        x = self.up2(x)
        x = self.dec3(x)
        x = self.dec4(x)

        # Output
        x = torch.sigmoid(self.out_conv(x))
        return x

# Model’i oluşturup CUDA’ya taşı
model = ColorizationCNN().to(device)
print(model)


ColorizationCNN(
  (conv1): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
  )
  (conv2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
  )
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv4): Sequential(
    (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
  )
  (conv5): Sequential(
    (0): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
  )
  (up1): Upsample(scale_factor=2.0, mode='nearest')
  (dec1): Sequential(
    (0): Conv2d(512, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    

In [4]:
# Hücre 4: Kayıp Fonksiyonu ve Optimizatör
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# Hücre 5: Eğitim ve Doğrulama Döngüleri
NUM_EPOCHS = 30
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS+1):
    # --- Eğitim ---
    model.train()
    running_loss = 0.0
    for gray, color in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]"):
        gray, color = gray.to(device), color.to(device)
        optimizer.zero_grad()
        output = model(gray)
        loss = criterion(output, color)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * gray.size(0)
    epoch_train_loss = running_loss / len(train_ds)
    train_losses.append(epoch_train_loss)

    # --- Doğrulama ---
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for gray, color in val_loader:
            gray, color = gray.to(device), color.to(device)
            output = model(gray)
            running_loss += criterion(output, color).item() * gray.size(0)
    epoch_val_loss = running_loss / len(val_ds)
    val_losses.append(epoch_val_loss)

    # Checkpoint
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch:02d} → Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")


Epoch 1/30 [Train]:   0%|          | 0/141 [00:00<?, ?it/s]

In [ ]:
# Hücre 6: Kayıpları Görselleştirme
plt.figure(figsize=(6,4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

In [ ]:
# Hücre 7: Test Seti Üzerinde PSNR ve SSIM Hesaplama
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

psnr_vals, ssim_vals = [], []
with torch.no_grad():
    for gray, color in tqdm(test_loader, desc="Test Evaluation"):
        gray_np  = gray.cpu().numpy()       # (B,1,128,128)
        color_np = color.cpu().numpy()      # (B,3,128,128)
        out = model(gray.to(device)).cpu().numpy()
        # Kanallar yeri değiştir, [B, H, W, C]
        out  = np.transpose(out,  (0,2,3,1))
        col  = np.transpose(color_np, (0,2,3,1))
        for i in range(out.shape[0]):
            psnr_vals.append( compute_psnr(col[i], out[i], data_range=1.0) )
            ssim_vals.append( compute_ssim(col[i], out[i], multichannel=True, data_range=1.0) )

print(f"Average PSNR: {np.mean(psnr_vals):.2f} dB")
print(f"Average SSIM: {np.mean(ssim_vals):.4f}")

In [ ]:
# Hücre 8: Örnek Sonuçları Gösterme
import random

def show_examples(n=3):
    model.eval()
    gray_batch, color_batch = next(iter(test_loader))
    idxs = random.sample(range(gray_batch.size(0)), n)
    with torch.no_grad():
        out_batch = model(gray_batch.to(device)).cpu()
    for i in idxs:
        gray = gray_batch[i,0].numpy()
        true = color_batch[i].permute(1,2,0).numpy()
        pred = out_batch[i].permute(1,2,0).numpy()

        fig, axes = plt.subplots(1,3, figsize=(12,4))
        axes[0].imshow(gray, cmap='gray'); axes[0].set_title("Gray Input")
        axes[1].imshow(true);           axes[1].set_title("Ground Truth")
        axes[2].imshow(pred);           axes[2].set_title("Predicted")
        for ax in axes: ax.axis('off')
        plt.show()

show_examples(n=5)